### Bakehouse dataset discovery

Objectifs : 
- connaître les six tables avant toute transformation ;
- identifier leur granularité, leurs clés et leurs relations

In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("SHOW TABLES IN samples.bakehouse").display()

In [0]:
tables = [
    "sales_transactions",
    "sales_customers",
    "sales_franchises",
    "sales_suppliers",
    "media_customer_reviews",
    "media_gold_reviews_chunked"
]

In [0]:
# Relever les colonnes et types 
# Compter les lignes de chaque tables 
for t in tables :
    df = spark.read.table(f"samples.bakehouse.{t}")
    print(f"table name: {t}, row count: {df.count()}, column count: {len(df.columns)}")
    df.printSchema()
    print("-"*20)

In [0]:
# Afficher les 10 premières lignes de chaque table
for t in tables :
    df = spark.read.table(f"samples.bakehouse.{t}")
    print(t)
    df.show(10, truncate=False)
    print("-"*20)

In [0]:
for t in tables : 
    df = spark.read.table(f"samples.bakehouse.{t}")
    cols = df.columns
    # Relever les plages de date pour les tables qui contiennent les dates
    col_date = [c for c in cols if "date" in c.lower()]
    for c in col_date : 
        plage_date = df.select(
            F.min(F.to_date(c)).alias("date_min"),
            F.max(F.to_date(c)).alias("date_max")
        ).first() 
        print(f"table name: {t}, column name: {c}, date range: {plage_date.date_min} - {plage_date.date_max}")

    # Relever les colonnes qui contiennent les valeurs nulles 
    for c in cols:
        null_count = df.agg(
        *[
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)]
    ).first()
        
        if null_count[c] > 0:
            print(f"table name: {t}, column name: {c}, null count: {null_count}")
            print("-"*20) 


        

##### Déterminer si la relation est 1–1, 1–N ou N–N

In [0]:
%sql
SELECT customerID, COUNT(*) as count_transactions
FROM samples.bakehouse.sales_transactions
WHERE customerID IS NOT NULL
GROUP BY customerID
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT franchiseID, COUNT(*) as count_transactions
FROM samples.bakehouse.sales_transactions
WHERE franchiseID IS NOT NULL
GROUP BY franchiseID
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT supplierID, COUNT(*) as count_franchises
FROM samples.bakehouse.sales_franchises
WHERE supplierID IS NOT NULL
GROUP BY supplierID
HAVING COUNT(*)>1

In [0]:
%sql
SELECT franchiseID, COUNT(*) as count_reviews
FROM samples.bakehouse.media_customer_reviews
WHERE franchiseID IS NOT NULL
GROUP BY franchiseID
HAVING COUNT(*)>1

In [0]:
%sql
SELECT franchiseID, COUNT(*) as count_reviews
FROM samples.bakehouse.media_gold_reviews_chunked
WHERE franchiseID IS NOT NULL
GROUP BY franchiseID
HAVING COUNT(*)>1

- La relation entre _sales_customers_ et _sales_transactions_ sont **1-N**
- La relation entre _sales_franchises_ et _sales_transactions_ est **1-N**
- La relation entre _sales_suppliers_ et _sales_franchises_ est **1-1** 
- La relation entre _media_customers_reviews_ et _sales_franchises_ est **N-1**
- La relation entre _media_golden_review_chunked_ et _sales_franchises_ est **N-1**

##### Chercher les clés orphelines avec un LEFT ANTI JOIN

In [0]:
%sql
SELECT t.*
FROM samples.bakehouse.sales_transactions t
LEFT ANTI JOIN samples.bakehouse.sales_customers c ON t.customerID = c.customerID
WHERE t.customerID IS NULL

In [0]:
%sql
SELECT t.*
FROM samples.bakehouse.sales_transactions t
LEFT ANTI JOIN samples.bakehouse.sales_franchises c ON t.franchiseID = c.franchiseID
WHERE t.franchiseID IS NULL

In [0]:
%sql
SELECT f.*
FROM samples.bakehouse.sales_franchises f
LEFT ANTI JOIN samples.bakehouse.sales_suppliers s ON f.supplierID = s.supplierID
WHERE f.supplierID IS NULL

In [0]:
%sql
SELECT mc.*
FROM samples.bakehouse.media_customer_reviews mc
LEFT ANTI JOIN samples.bakehouse.sales_franchises f ON mc.franchiseID = f.franchiseID
WHERE mc.franchiseID IS NULL

In [0]:
%sql
SELECT mg.*
FROM samples.bakehouse.media_gold_reviews_chunked mg
LEFT ANTI JOIN samples.bakehouse.sales_franchises f ON mg.franchiseID = f.franchiseID
WHERE mg.franchiseID IS NULL

- Les customerID dans _sales_customers_ sont présents dans _sales_transactions_
- Les franchiseID dans _sales_franchises_ sont présents dans _sales_transactions_, _sales_suppliers_, _media_customers_reviews_ et _media_gold_reviews_chunked_